<div align="center">

# RouteFM: Pretrain Once, Route Anywhere

### Route a new query from behavioral context—without retraining

[Project page](https://lamda-model-reuse.github.io/RouteFM/) · [Paper](https://arxiv.org/abs/2609.37362) · [GitHub](https://github.com/LAMDA-Model-Reuse/RouteFM) · [Models](https://huggingface.co/AIGNLAI/RouteFM) · [Demo](https://huggingface.co/spaces/AIGNLAI/RouteFM-Demo)

</div>

This notebook runs the released **RouteFM-BGE** checkpoint on an illustrative candidate pool. It works on a free Colab CPU runtime: choose **Runtime → Run all**, then edit the marked configuration cell and rerun from there.

> The built-in behavioral observations are synthetic and are not benchmark records. Candidate names only label the output; RouteFM does not use them as input features.

## 1. Install the published package

The package downloads immutable, checksum-verified RouteFM weights from Hugging Face on first use. The external BGE query encoder is frozen.

In [ ]:
%pip -q install "routefm-router[bge]==1.1.0" matplotlib pandas

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from routefm import RouteFMRouter

router = RouteFMRouter.from_pretrained(
    encoder="bge",
    device="cpu",
    embedding_batch_size=32,
    embedding_max_length=256,
)
print("RouteFM-BGE is ready.")

## 2. Define an illustrative routing environment

Every observation is a `(query, quality, relative cost)` tuple. Quality must lie in `[0, 1]`; cost must be nonnegative. Costs only need to be comparable inside the current environment.

In [ ]:
CANDIDATE_PROFILES = [
    "Fast generalist",
    "Reasoning specialist",
    "Code specialist",
    "Broad premium model",
]

# query, per-candidate quality scores, per-candidate relative costs
OBSERVATIONS = [
    ("Summarize the causes of the 2008 financial crisis.", [0.76, 0.80, 0.62, 0.91], [0.08, 0.38, 0.22, 0.92]),
    ("Solve: if 3x + 7 = 25, find x.", [0.88, 0.98, 0.70, 0.98], [0.06, 0.34, 0.20, 0.88]),
    ("Write a Python function for binary search.", [0.71, 0.75, 0.98, 0.94], [0.09, 0.40, 0.24, 0.95]),
    ("Explain why the sky appears blue.", [0.84, 0.89, 0.72, 0.95], [0.07, 0.36, 0.21, 0.90]),
    ("Find the derivative of x^3 sin(x).", [0.72, 0.97, 0.78, 0.98], [0.07, 0.36, 0.22, 0.91]),
    ("Debug a race condition in a producer-consumer queue.", [0.52, 0.70, 0.96, 0.93], [0.10, 0.44, 0.27, 1.00]),
    ("Draft a concise, polite meeting follow-up email.", [0.91, 0.84, 0.69, 0.94], [0.06, 0.35, 0.20, 0.87]),
    ("Compare utilitarian and deontological ethics.", [0.74, 0.86, 0.60, 0.95], [0.08, 0.40, 0.23, 0.96]),
]

def make_context(k: int, reveal_labels: bool = False):
    if k not in {1, 2, 4, 8}:
        raise ValueError("K must be one of 1, 2, 4, or 8")
    names = (
        CANDIDATE_PROFILES
        if reveal_labels
        else [f"Candidate {chr(65 + index)}" for index in range(len(CANDIDATE_PROFILES))]
    )
    return {
        name: [
            {"query": query, "score": scores[index], "cost": costs[index]}
            for query, scores, costs in OBSERVATIONS[:k]
        ]
        for index, name in enumerate(names)
    }

context_preview = pd.DataFrame([
    {"query": query, **{f"Candidate {chr(65+i)} score": score for i, score in enumerate(scores)}}
    for query, scores, _ in OBSERVATIONS
])
display(context_preview)

## 3. Route a target query

Edit the three values below. Increasing `K` gives RouteFM more evidence about each candidate. Revealing labels changes presentation only—it does not change the router inputs.

In [ ]:
#@title Routing configuration
TARGET_QUERY = "Prove that there are infinitely many prime numbers." #@param {type:"string"}
K = 4 #@param [1, 2, 4, 8] {type:"raw"}
REVEAL_DESCRIPTIVE_LABELS = False #@param {type:"boolean"}

context = make_context(K, REVEAL_DESCRIPTIVE_LABELS)
decision = router.set_context(context).route(TARGET_QUERY)
print(f"Selected: {decision.model_name}")

In [ ]:
result = pd.DataFrame({
    "candidate": list(decision.predicted_scores),
    "predicted_quality": list(decision.predicted_scores.values()),
    "predicted_relative_cost": list(decision.predicted_relative_costs.values()),
})
result["selected"] = result["candidate"] == decision.model_name
display(result.style.format({"predicted_quality": "{:.3f}", "predicted_relative_cost": "{:.3f}"}))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
colors = ["#605bf6" if value else "#b9bec8" for value in result["selected"]]
axes[0].barh(result["candidate"], result["predicted_quality"], color=colors)
axes[0].set_title("Predicted quality")
axes[0].set_xlim(0, 1)
axes[1].barh(result["candidate"], result["predicted_relative_cost"], color="#41c7b7")
axes[1].set_title("Predicted relative cost")
axes[1].set_xlim(0, 1)
for axis in axes:
    axis.grid(axis="x", alpha=0.2)
    axis.set_axisbelow(True)
plt.suptitle(f"RouteFM-BGE · K={K} · selected {decision.model_name}", fontweight="bold")
plt.tight_layout()
plt.show()

print("Decision rule: highest predicted quality. Relative cost is predicted but is not used by the default rule.")

## 4. See how the context budget changes predictions

The same frozen router is reused at every `K`; no parameters are updated.

In [ ]:
sweep_rows = []
for budget in (1, 2, 4, 8):
    local = router.set_context(make_context(budget)).route(TARGET_QUERY)
    for candidate, score in local.predicted_scores.items():
        sweep_rows.append({
            "K": budget,
            "candidate": candidate,
            "predicted_quality": score,
            "selected": candidate == local.model_name,
        })

sweep = pd.DataFrame(sweep_rows)
display(sweep.pivot(index="K", columns="candidate", values="predicted_quality").style.format("{:.3f}"))

fig, axis = plt.subplots(figsize=(9, 4.5))
for candidate, rows in sweep.groupby("candidate"):
    axis.plot(rows["K"], rows["predicted_quality"], marker="o", label=candidate)
axis.set(xlabel="Observations per candidate (K)", ylabel="Predicted quality", xticks=[1, 2, 4, 8], ylim=(0, 1))
axis.grid(alpha=0.2)
axis.legend(ncol=2, frameon=False)
axis.set_title("One frozen RouteFM, changing behavioral evidence", fontweight="bold")
plt.show()

## 5. Bring your own candidate context

Replace the example below with observations collected from your candidate models. Candidates may have different numbers of observations. RouteFM predicts routing quality and relative cost; it does not call the candidate models or generate their answers.

In [ ]:
CUSTOM_CONTEXT = {
    "small-model": [
        {"query": "What is 2 + 2?", "score": 1.0, "cost": 0.01},
        {"query": "Summarize a news article.", "score": 0.72, "cost": 0.02},
    ],
    "large-model": [
        {"query": "What is 2 + 2?", "score": 1.0, "cost": 0.10},
        {"query": "Summarize a news article.", "score": 0.95, "cost": 0.20},
    ],
}
CUSTOM_QUERY = "Explain the intuition behind the central limit theorem."

custom_decision = router.set_context(CUSTOM_CONTEXT).route(CUSTOM_QUERY)
custom_result = pd.DataFrame({
    "candidate": list(custom_decision.predicted_scores),
    "predicted_quality": list(custom_decision.predicted_scores.values()),
    "predicted_relative_cost": list(custom_decision.predicted_relative_costs.values()),
})
print(f"Selected: {custom_decision.model_name}")
display(custom_result.style.format({"predicted_quality": "{:.3f}", "predicted_relative_cost": "{:.3f}"}))

## 6. Export the decision

The saved JSON records the query and all per-candidate predictions, making it easy to feed the decision into a serving layer.

In [ ]:
export = {"query": CUSTOM_QUERY, **custom_decision.to_dict()}
output_path = Path("/content/routefm_decision.json")
output_path.write_text(json.dumps(export, indent=2) + "\n", encoding="utf-8")
print(json.dumps(export, indent=2))
print(f"\nSaved to {output_path}")

## Interpretation and limitations

- The default decision rule chooses the highest predicted quality; predicted relative cost is not a calibrated monetary or latency estimate.
- RouteFM assumes every candidate has at least one valid behavioral observation.
- Performance may change across languages, domains, candidate pools, or encoder revisions outside the training distribution.
- Candidate labels are for presentation only; identities, providers, and parameter counts are not router features.

If you use RouteFM in research, please cite:

```bibtex
@misc{lai2026pretrainoncerouteanywhere,
  title         = {Pretrain Once, Route Anywhere: Towards a Foundation Model for LLM Routing},
  author        = {Guannan Lai and Han-Jia Ye},
  year          = {2026},
  eprint        = {2609.37362},
  archivePrefix = {arXiv},
  primaryClass  = {cs.AI}
}
```